# Notebook CKAN Registration

This notebook creates or refreshes a CKAN dataset entry from the contents of a Jupyter notebook.

Workflow:
1. Read a target `.ipynb` file.
2. Walk through every source line, preserving cell and line references.
3. Ask an LLM to recommend which context is useful for CKAN metadata.
4. Combine the recommended context into one compact evidence packet.
5. Ask a second LLM call to create CKAN-ready dataset metadata.
6. If an existing CKAN entry is provided, update it. Otherwise, create a new dataset when `APPLY_CHANGES=True`.
7. Optionally upload the notebook itself as a CKAN resource.

Safety defaults:
- `APPLY_CHANGES = False`
- `UPLOAD_NOTEBOOK_RESOURCE = False`


In [ ]:
import json
import os
import re
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote, urlparse

from IPython.display import Markdown, display
from dotenv import load_dotenv

# Bootstrap: add src/ to sys.path so gam_registration package is importable.
# When running from notebooks/, the repo root is one level up.
_NB_DIR = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
_REPO_ROOT = _NB_DIR.parent if (_NB_DIR.parent / "src").exists() else _NB_DIR
if str(_REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT / "src"))

from gam_registration import utils as u

WORKFLOW_DIR = _REPO_ROOT
load_dotenv(WORKFLOW_DIR / ".env")


def show_md(text: str) -> None:
    display(Markdown(text))


show_md(f"""
**Environment Ready**
- Standalone notebook registration workflow
- Workflow directory: `{WORKFLOW_DIR}`
- Uses local `utils.py` for CKAN auth, LLM calls, and dataset create/update
""")

## 1) Configuration

Set `NOTEBOOK_PATH` to the notebook you want to register. If `EXISTING_CKAN_ENTRY` is set to a CKAN dataset name, ID, or dataset URL, the workflow updates that dataset name. If it is empty, the metadata LLM proposes a new dataset name.


In [ ]:
def resolve_workflow_path(value: str) -> Path:
    path = Path(value).expanduser()
    if not path.is_absolute():
        path = (WORKFLOW_DIR / path).resolve()
    return path


NOTEBOOK_PATH = resolve_workflow_path(os.getenv("NOTEBOOK_PATH", "archive/Opera- Subsidence.ipynb"))
EXISTING_CKAN_ENTRY = os.getenv("EXISTING_CKAN_ENTRY", "houston-opera-subsidence-estimates").strip() or None
LEGACY_SOURCE_METADATA_URL = os.getenv("SOURCE_METADATA_URL", "").strip()
BLOCKED_DATASET_URLS = {
    LEGACY_SOURCE_METADATA_URL,
    "https://www.twdb.texas.gov/groundwater/models/gam/ygjk/ygjk.asp",
}
BLOCKED_DATASET_URLS = {value.rstrip("/") for value in BLOCKED_DATASET_URLS if value}


def is_blocked_dataset_url(value: str | None) -> bool:
    url = re.sub(r"\s+", " ", str(value or "")).strip().rstrip("/")
    if not url:
        return False
    return url in BLOCKED_DATASET_URLS or "twdb.texas.gov/groundwater/models/gam/ygjk" in url.lower()


def git_stdout(args: list[str], cwd: Path) -> str:
    try:
        result = subprocess.run(
            ["git", *args],
            cwd=str(cwd),
            capture_output=True,
            check=True,
            text=True,
            timeout=10,
        )
    except Exception:
        return ""
    return result.stdout.strip()


def github_remote_base_url(remote_url: str) -> str:
    remote_url = re.sub(r"\s+", " ", str(remote_url or "")).strip()
    if not remote_url:
        return ""
    if remote_url.startswith("git@github.com:"):
        repo_path = remote_url.split(":", 1)[1]
        return "https://github.com/" + repo_path.removesuffix(".git")
    parsed = urlparse(remote_url)
    if parsed.netloc.lower() == "github.com":
        return f"https://github.com{parsed.path.removesuffix('.git')}".rstrip("/")
    return ""


def infer_github_blob_url(path: Path) -> str:
    if not path.exists():
        return ""
    repo_root_text = git_stdout(["rev-parse", "--show-toplevel"], path.parent)
    if not repo_root_text:
        return ""
    repo_root = Path(repo_root_text)
    remote_url = git_stdout(["remote", "get-url", "origin"], repo_root)
    github_base = github_remote_base_url(remote_url)
    if not github_base:
        return ""
    ref = git_stdout(["rev-parse", "--abbrev-ref", "HEAD"], repo_root)
    if not ref or ref == "HEAD":
        ref = git_stdout(["rev-parse", "HEAD"], repo_root)
    if not ref:
        return ""
    try:
        relative_path = path.resolve().relative_to(repo_root.resolve()).as_posix()
    except ValueError:
        return ""
    return f"{github_base}/blob/{quote(ref, safe='')}/{quote(relative_path, safe='/')}"


NOTEBOOK_SOURCE_URL = os.getenv("NOTEBOOK_SOURCE_URL", "").strip()
if is_blocked_dataset_url(NOTEBOOK_SOURCE_URL):
    NOTEBOOK_SOURCE_URL = ""
NOTEBOOK_SOURCE_URL = NOTEBOOK_SOURCE_URL or infer_github_blob_url(NOTEBOOK_PATH)
if is_blocked_dataset_url(NOTEBOOK_SOURCE_URL):
    NOTEBOOK_SOURCE_URL = ""

PREFERRED_DATASET_NAME = os.getenv("NOTEBOOK_CKAN_DATASET_NAME", "").strip() or None
PREFERRED_DATASET_TITLE = os.getenv("NOTEBOOK_CKAN_DATASET_TITLE", "").strip() or None

CKAN_URL = os.getenv("CKAN_URL", "https://ckan.tacc.utexas.edu").strip()
CKAN_OWNER_ORG = os.getenv("CKAN_OWNER_ORG", "dso-institute").strip()
CKAN_OWNER_ORG_ID = os.getenv("CKAN_OWNER_ORG_ID", "2f68b69f-95b8-468c-b0c0-39d916f26c61").strip()
CKAN_AUTH_MODE = os.getenv("CKAN_AUTH_MODE", "tapis_password").strip()
CKAN_API_TOKEN = os.getenv("CKAN_API_TOKEN", os.getenv("CKAN_API_KEY", "")).strip()
CKAN_USERNAME = os.getenv("CKAN_USERNAME", "").strip()
CKAN_PASSWORD = os.getenv("CKAN_PASSWORD", "")
CKAN_TAPIS_URL = os.getenv("CKAN_TAPIS_URL", u.DEFAULT_TAPIS_URL).strip()

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://ai.tejas.tacc.utexas.edu").strip() or None
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
CKAN_LLM_MODEL = os.getenv("CKAN_LLM_MODEL", os.getenv("TOPIC_LABELER_MODEL", "Meta-Llama-3.3-70B-Instruct")).strip()

LINES_PER_CONTEXT_CHUNK = int(os.getenv("LINES_PER_CONTEXT_CHUNK", "120"))
MAX_CONTEXT_CHUNKS = int(os.getenv("MAX_CONTEXT_CHUNKS", "0")) or None
MAX_CONTEXT_ITEMS = int(os.getenv("MAX_CONTEXT_ITEMS", "120"))
MAX_COMBINED_CONTEXT_CHARS = int(os.getenv("MAX_COMBINED_CONTEXT_CHARS", "28000"))

APPLY_CHANGES = False
UPLOAD_NOTEBOOK_RESOURCE = False
FORCE_DATASET_METADATA_UPDATE = True

CKAN_DATASET_LICENSE_ID = os.getenv("CKAN_DATASET_LICENSE_ID", "cc-by").strip() or None
CKAN_DATASET_MAINTAINER = os.getenv("CKAN_DATASET_MAINTAINER", "William Mobley").strip() or None
CKAN_DATASET_MAINTAINER_EMAIL = os.getenv("CKAN_DATASET_MAINTAINER_EMAIL", "wmobley@tacc.utexas.edu").strip() or None
CKAN_DATASET_AUTHOR = os.getenv("CKAN_DATASET_AUTHOR", "").strip() or None
CKAN_DATASET_AUTHOR_EMAIL = os.getenv("CKAN_DATASET_AUTHOR_EMAIL", "").strip() or None
CKAN_DATASET_VERSION = os.getenv("CKAN_DATASET_VERSION", "").strip() or None
CKAN_DATASET_TYPE = os.getenv("CKAN_DATASET_TYPE", "dataset").strip() or "dataset"
CKAN_DATASET_ISOPEN = os.getenv("CKAN_DATASET_ISOPEN", "true").strip().lower() in {"1", "true", "yes"}
CKAN_DATASET_SPATIAL = os.getenv("CKAN_DATASET_SPATIAL", "").strip() or None
CKAN_TEMPORAL_COVERAGE_START = os.getenv("CKAN_TEMPORAL_COVERAGE_START", "").strip() or None
CKAN_TEMPORAL_COVERAGE_END = os.getenv("CKAN_TEMPORAL_COVERAGE_END", "").strip() or None

show_md(f"""
## Configuration Summary
- **NOTEBOOK_PATH:** `{NOTEBOOK_PATH}`
- **EXISTING_CKAN_ENTRY:** `{EXISTING_CKAN_ENTRY or '<none>'}`
- **PREFERRED_DATASET_NAME:** `{PREFERRED_DATASET_NAME or '<LLM/generated>'}`
- **PREFERRED_DATASET_TITLE:** `{PREFERRED_DATASET_TITLE or '<LLM/generated>'}`
- **NOTEBOOK_SOURCE_URL:** `{NOTEBOOK_SOURCE_URL or '<none/inferred unavailable>'}`
- **Blocked dataset URL count:** `{len(BLOCKED_DATASET_URLS)}`
- **CKAN_URL:** `{CKAN_URL}`
- **CKAN_OWNER_ORG:** `{CKAN_OWNER_ORG}`
- **CKAN_OWNER_ORG_ID:** `{CKAN_OWNER_ORG_ID}`
- **LLM model:** `{CKAN_LLM_MODEL}`
- **OPENAI_API_KEY set:** `{bool(OPENAI_API_KEY)}`
- **LINES_PER_CONTEXT_CHUNK:** `{LINES_PER_CONTEXT_CHUNK}`
- **MAX_CONTEXT_CHUNKS:** `{MAX_CONTEXT_CHUNKS or '<all>'}`
- **APPLY_CHANGES:** `{APPLY_CHANGES}`
- **UPLOAD_NOTEBOOK_RESOURCE:** `{UPLOAD_NOTEBOOK_RESOURCE}`
""")

## 2) Helpers


In [ ]:
clean_text = u.clean_text
slugify = u.slugify
dedupe_tags = u.dedupe_tags
sanitize_tag = u.sanitize_tag
fetch_existing_dataset_or_none = u.fetch_existing_dataset_or_none
compare_dataset_metadata = u.compare_dataset_metadata
render_changes_table_markdown = u.render_changes_table_markdown
upsert_resources = u.upsert_resources


def load_notebook(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Notebook does not exist: {path}")
    return json.loads(path.read_text(encoding="utf-8"))


def flatten_output_value(value: Any) -> str:
    if value is None:
        return ""
    if isinstance(value, list):
        return "".join(str(item) for item in value)
    if isinstance(value, (dict, tuple)):
        return json.dumps(value, sort_keys=True)
    return str(value)


def output_text_lines(output: dict[str, Any]) -> list[str]:
    output_type = clean_text(output.get("output_type"))
    chunks: list[str] = []

    if output_type == "stream":
        chunks.append(flatten_output_value(output.get("text")))
    elif output_type in {"display_data", "execute_result"}:
        data = output.get("data") or {}
        for mime_type in ["text/plain", "text/markdown", "application/json"]:
            if mime_type in data:
                chunks.append(flatten_output_value(data.get(mime_type)))
        if "text/html" in data and not chunks:
            html_text = re.sub(r"<script[\s\S]*?</script>", " ", flatten_output_value(data.get("text/html")), flags=re.IGNORECASE)
            html_text = re.sub(r"<style[\s\S]*?</style>", " ", html_text, flags=re.IGNORECASE)
            html_text = re.sub(r"<[^>]+>", " ", html_text)
            chunks.append(html_text)
    elif output_type == "error":
        chunks.append(" ".join(clean_text(part) for part in output.get("traceback", [])))
        chunks.append(f"{clean_text(output.get('ename'))}: {clean_text(output.get('evalue'))}")

    lines: list[str] = []
    for chunk in chunks:
        chunk = re.sub(r"\x1b\[[0-9;]*m", "", chunk)
        for line in chunk.splitlines():
            cleaned = clean_text(line, 1200)
            if cleaned:
                lines.append(cleaned)
    return lines


def notebook_line_records(notebook: dict[str, Any]) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    for cell_index, cell in enumerate(notebook.get("cells", [])):
        cell_type = clean_text(cell.get("cell_type"))
        source = cell.get("source", "")
        if isinstance(source, list):
            text = "".join(str(item) for item in source)
        else:
            text = str(source or "")

        for line_number, line in enumerate(text.splitlines(), start=1):
            records.append(
                {
                    "cell_index": cell_index,
                    "cell_type": cell_type,
                    "record_kind": "source",
                    "line_number": line_number,
                    "text": line.rstrip(),
                }
            )

        for output_index, output in enumerate(cell.get("outputs", []) or [], start=1):
            for output_line_number, line in enumerate(output_text_lines(output), start=1):
                records.append(
                    {
                        "cell_index": cell_index,
                        "cell_type": cell_type,
                        "record_kind": "output",
                        "output_index": output_index,
                        "output_type": clean_text(output.get("output_type")),
                        "output_line_number": output_line_number,
                        "text": line,
                    }
                )
    return records


def notebook_source_records(notebook: dict[str, Any]) -> list[dict[str, Any]]:
    return notebook_line_records(notebook)


def chunk_records(records: list[dict[str, Any]], lines_per_chunk: int) -> list[list[dict[str, Any]]]:
    return [records[start:start + lines_per_chunk] for start in range(0, len(records), lines_per_chunk)]


def line_ref(record: dict[str, Any]) -> str:
    if record.get("record_kind") == "output":
        return f"cell {record['cell_index']} output {record.get('output_index', 0)}:{record.get('output_line_number', 0)}"
    return f"cell {record['cell_index']}:{record['line_number']}"


def chunk_to_numbered_text(chunk: list[dict[str, Any]], max_chars: int = 12000) -> str:
    lines = []
    for record in chunk:
        text = record["text"]
        if len(text) > 500:
            text = text[:497].rstrip() + "..."
        kind = record.get("record_kind", "source")
        lines.append(f"{line_ref(record)} [{record['cell_type']} {kind}]: {text}")
    result = "\n".join(lines)
    if len(result) > max_chars:
        return result[:max_chars - 3].rstrip() + "..."
    return result


def first_markdown_heading(records: list[dict[str, Any]]) -> str:
    for record in records:
        if record.get("record_kind") == "source" and record["cell_type"] == "markdown":
            match = re.match(r"^#\s+(.+)", record["text"].strip())
            if match:
                return clean_text(match.group(1), 140)
    return ""


def collect_notebook_inventory(path: Path, notebook: dict[str, Any], records: list[dict[str, Any]]) -> dict[str, Any]:
    cells = notebook.get("cells", [])
    headings = []
    imports = []
    urls = []
    file_mentions = []
    temporal_hints = []
    spatial_hints = []
    unit_hints = []
    source_line_count = sum(1 for record in records if record.get("record_kind") == "source")
    output_line_count = sum(1 for record in records if record.get("record_kind") == "output")

    date_pattern = re.compile(r"\b(?:\d{4}-\d{1,2}-\d{1,2}|\d{4}/\d{1,2}/\d{1,2}|\d{4})\b")
    temporal_pattern = re.compile(r"\b(?:date|time|temporal|period|coverage|start|end|min|max|year|epoch)\b", re.IGNORECASE)
    spatial_pattern = re.compile(r"\b(?:bbox|bounds?|bounding|extent|polygon|geojson|geometry|aoi|area of interest|longitude|latitude|lon|lat|xlim|ylim|epsg|crs)\b", re.IGNORECASE)
    unit_pattern = re.compile(r"\b(?:units?|unit_of_measure|measurement|measure|mm|millimeter|millimetre|cm|centimeter|metre|meter|m/year|mm/year|cm/year|in/year|feet|ft|subsidence|compaction|displacement|velocity|rate)\b", re.IGNORECASE)

    for record in records:
        text = record["text"].strip()
        if not text:
            continue
        if record.get("record_kind") == "source" and record["cell_type"] == "markdown" and re.match(r"^#{1,4}\s+", text):
            headings.append({"ref": line_ref(record), "text": clean_text(text, 220)})
        if record.get("record_kind") == "source" and record["cell_type"] == "code" and (text.startswith("import ") or text.startswith("from ")):
            imports.append({"ref": line_ref(record), "text": clean_text(text, 220)})
        for url in re.findall(r"https?://[^\s\)\]\x27\"]+", text):
            urls.append({"ref": line_ref(record), "url": url.rstrip('.,')})
        for mention in re.findall(r"[A-Za-z0-9_./-]+\.(?:csv|json|geojson|tif|tiff|html|ipynb|py|nc|zip|parquet)", text):
            file_mentions.append({"ref": line_ref(record), "file": mention})
        if temporal_pattern.search(text) and date_pattern.search(text):
            temporal_hints.append({"ref": line_ref(record), "text": clean_text(text, 260)})
        if spatial_pattern.search(text):
            spatial_hints.append({"ref": line_ref(record), "text": clean_text(text, 320)})
        if unit_pattern.search(text):
            unit_hints.append({"ref": line_ref(record), "text": clean_text(text, 260)})

    kernelspec = notebook.get("metadata", {}).get("kernelspec", {})
    return {
        "path": str(path),
        "file_name": path.name,
        "file_stem": path.stem,
        "title_hint": first_markdown_heading(records) or path.stem.replace("_", " ").replace("-", " ").title(),
        "cell_count": len(cells),
        "code_cell_count": sum(1 for cell in cells if cell.get("cell_type") == "code"),
        "markdown_cell_count": sum(1 for cell in cells if cell.get("cell_type") == "markdown"),
        "line_count": len(records),
        "source_line_count": source_line_count,
        "output_line_count": output_line_count,
        "kernel": kernelspec.get("display_name") or kernelspec.get("name") or "",
        "headings": headings[:80],
        "imports": imports[:80],
        "urls": urls[:80],
        "file_mentions": file_mentions[:100],
        "temporal_hints": temporal_hints[:80],
        "spatial_hints": spatial_hints[:80],
        "unit_hints": unit_hints[:80],
    }

def normalize_existing_ckan_entry(value: str | None) -> str | None:
    value = clean_text(value)
    if not value:
        return None
    parsed = urlparse(value)
    if parsed.scheme and parsed.netloc:
        parts = [part for part in parsed.path.split("/") if part]
        if "dataset" in parts:
            index = parts.index("dataset")
            if index + 1 < len(parts):
                return parts[index + 1]
    return value


def compact_existing_dataset(dataset: dict[str, Any] | None) -> dict[str, Any] | None:
    if not dataset:
        return None
    return {
        "id": dataset.get("id"),
        "name": dataset.get("name"),
        "title": dataset.get("title"),
        "notes": clean_text(dataset.get("notes"), 1200),
        "url": dataset.get("url"),
        "author": dataset.get("author"),
        "author_email": dataset.get("author_email"),
        "maintainer": dataset.get("maintainer"),
        "maintainer_email": dataset.get("maintainer_email"),
        "license_id": dataset.get("license_id"),
        "version": dataset.get("version"),
        "type": dataset.get("type"),
        "isopen": dataset.get("isopen"),
        "spatial": dataset.get("spatial"),
        "temporal_coverage_start": dataset.get("temporal_coverage_start"),
        "temporal_coverage_end": dataset.get("temporal_coverage_end"),
        "tags": [tag.get("name") for tag in dataset.get("tags", []) if tag.get("name")],
        "resource_names": [resource.get("name") for resource in dataset.get("resources", []) if resource.get("name")],
    }


def llm_json(system_prompt: str, user_payload: dict[str, Any], *, temperature: float = 0.1) -> dict[str, Any]:
    content = u._chat_completion_content(
        model=CKAN_LLM_MODEL,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
        system_prompt=system_prompt,
        user_payload=user_payload,
        temperature=temperature,
    )
    parsed = u._parse_llm_json(content)
    if not parsed:
        raise ValueError(f"LLM did not return parseable JSON. Raw content: {clean_text(content, 1000)}")
    return parsed


def render_context_items_table(context_items: list[dict[str, Any]], limit: int = 30) -> str:
    lines = [
        "| # | category | summary | evidence |",
        "|---:|---|---|---|",
    ]
    for idx, item in enumerate(context_items[:limit], start=1):
        lines.append(
            "| {idx} | {category} | {summary} | {evidence} |".format(
                idx=idx,
                category=clean_text(item.get("category"), 80).replace("|", "\\|"),
                summary=clean_text(item.get("summary"), 240).replace("|", "\\|"),
                evidence=clean_text(item.get("evidence"), 180).replace("|", "\\|"),
            )
        )
    return "\n".join(lines)

## 3) Load Notebook and Inventory Source Lines


In [ ]:
notebook = load_notebook(NOTEBOOK_PATH)
line_records = notebook_line_records(notebook)
line_chunks = chunk_records(line_records, LINES_PER_CONTEXT_CHUNK)
if MAX_CONTEXT_CHUNKS is not None:
    line_chunks = line_chunks[:MAX_CONTEXT_CHUNKS]

notebook_inventory = collect_notebook_inventory(NOTEBOOK_PATH, notebook, line_records)

show_md(f"""
## Notebook Inventory
- Path: `{NOTEBOOK_PATH}`
- Title hint: **{notebook_inventory['title_hint']}**
- Cells: **{notebook_inventory['cell_count']}** ({notebook_inventory['code_cell_count']} code, {notebook_inventory['markdown_cell_count']} markdown)
- Source lines walked: **{notebook_inventory['source_line_count']}**
- Output lines walked: **{notebook_inventory['output_line_count']}**
- Total context lines walked: **{notebook_inventory['line_count']}**
- Context chunks to review: **{len(line_chunks)}**
- Kernel: `{notebook_inventory['kernel'] or '<unknown>'}`
""")

if notebook_inventory["headings"]:
    show_md("\n".join(["**Heading Preview**"] + [f"- `{item['ref']}` {item['text']}" for item in notebook_inventory["headings"][:12]]))

if notebook_inventory["temporal_hints"]:
    show_md("\n".join(["**Temporal Hint Preview**"] + [f"- `{item['ref']}` {item['text']}" for item in notebook_inventory["temporal_hints"][:12]]))

## 4) Resolve CKAN Auth and Existing Dataset

`EXISTING_CKAN_ENTRY` may be a dataset name, UUID, or CKAN dataset URL. If it resolves, the generated metadata will preserve that package name and update the existing package when changes are applied.


In [ ]:
auth_header = None
normalized_auth_mode = CKAN_AUTH_MODE.lower()
if normalized_auth_mode == "api_token":
    auth_header = u.build_ckan_auth_header(
        auth_mode=CKAN_AUTH_MODE,
        api_token=CKAN_API_TOKEN,
        username=CKAN_USERNAME,
        password=CKAN_PASSWORD,
        tapis_url=CKAN_TAPIS_URL,
    )
elif normalized_auth_mode == "tapis_password":
    if APPLY_CHANGES:
        auth_header = u.build_ckan_auth_header(
            auth_mode=CKAN_AUTH_MODE,
            api_token=CKAN_API_TOKEN,
            username=CKAN_USERNAME,
            password=CKAN_PASSWORD,
            tapis_url=CKAN_TAPIS_URL,
        )
    else:
        show_md("Dry run: Tapis CKAN auth is deferred until `APPLY_CHANGES=True`; public CKAN reads will be attempted without auth.")
else:
    raise ValueError(f"Unsupported CKAN auth mode: {CKAN_AUTH_MODE}")

existing_lookup_id = normalize_existing_ckan_entry(EXISTING_CKAN_ENTRY)
previous_ckan_dataset = None
if existing_lookup_id:
    previous_ckan_dataset = fetch_existing_dataset_or_none(CKAN_URL, existing_lookup_id, auth_header)
    if previous_ckan_dataset is None:
        show_md(f"**Existing CKAN entry not found during lookup:** `{existing_lookup_id}`. The workflow will still preserve this identifier as the target dataset name if metadata is applied.")

show_md(f"""
## CKAN Target
- Auth mode: `{CKAN_AUTH_MODE}`
- Auth header ready: `{bool(auth_header)}`
- Existing lookup: `{existing_lookup_id or '<none>'}`
- Existing dataset found: `{bool(previous_ckan_dataset)}`
- Existing dataset name: `{previous_ckan_dataset.get('name') if previous_ckan_dataset else '<none>'}`
""")

## 5) LLM Pass 1: Recommend Useful CKAN Context From Notebook Lines

This pass reviews all notebook source lines in chunks. It does not create CKAN metadata yet; it only extracts evidence that should inform a CKAN entry.


In [ ]:
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required for notebook context extraction.")

CONTEXT_SYSTEM_PROMPT = """
You are reviewing Jupyter notebook source lines and text output lines to identify evidence for CKAN metadata.
The CKAN entry is for the output dataset or data product generated by the notebook, not for the notebook file itself.
Return STRICT JSON only with this shape:
{
  "context_items": [
    {
      "category": "purpose|data-source|method|output|spatial|temporal|software|organization|license|resource|other",
      "summary": "one concise sentence",
      "evidence": "line references and short evidence text",
      "confidence": "high|medium|low"
    }
  ]
}
Rules:
- Select only information useful for describing the dataset, map layer, file, API product, or other data output produced by the notebook.
- Treat the notebook path/name as provenance only; do not make the notebook itself the dataset unless the lines clearly say the notebook is the deliverable.
- Prefer evidence about generated outputs, source datasets, data transformations, spatial scope, temporal scope, units of measure, organizations, licenses, and publication/upload targets.
- For spatial context, actively look for bbox variables, map bounds, drawn AOI geometry, longitude/latitude extents, GeoJSON, CRS/EPSG, and output lines showing the actual selected area.
- If a bbox is present, preserve the coordinate order evidence. For common lon/lat bboxes, summarize west, south, east, north when supported.
- For temporal context, actively look for variable assignments, constants, dataframe filters, filename date ranges, date column min/max calculations, and printed/displayed output showing start/end dates or years.
- For measurement context, preserve units and measured quantities such as displacement, velocity, compaction, subsidence, elevation, depth, or rate when supported.
- If source variables and output prints disagree on temporal coverage, prefer printed/displayed output that reflects the actual executed result; otherwise preserve both with evidence.
- Preserve concrete URLs, dataset names, file names, layer names, variable names, source organizations, and date values when present.
- Do not include code mechanics unless they explain dataset content, provenance, spatial coverage, temporal coverage, or outputs.
- Do not invent context that is not supported by source or output lines.
- Keep at most 12 context_items for this chunk.
"""

context_items: list[dict[str, Any]] = []
for chunk_index, chunk in enumerate(line_chunks, start=1):
    payload = {
        "notebook_inventory": {
            "path": notebook_inventory["path"],
            "title_hint": notebook_inventory["title_hint"],
            "cell_count": notebook_inventory["cell_count"],
            "line_count": notebook_inventory["line_count"],
            "source_line_count": notebook_inventory["source_line_count"],
            "output_line_count": notebook_inventory["output_line_count"],
            "temporal_hints": notebook_inventory["temporal_hints"][:20],
            "spatial_hints": notebook_inventory["spatial_hints"][:20],
            "unit_hints": notebook_inventory["unit_hints"][:20],
            "kernel": notebook_inventory["kernel"],
        },
        "chunk_index": chunk_index,
        "chunk_count": len(line_chunks),
        "notebook_source_and_output_lines": chunk_to_numbered_text(chunk),
    }
    parsed = llm_json(CONTEXT_SYSTEM_PROMPT, payload, temperature=0.0)
    for item in parsed.get("context_items", []):
        if not isinstance(item, dict):
            continue
        summary = clean_text(item.get("summary"), 500)
        if not summary:
            continue
        context_items.append(
            {
                "category": clean_text(item.get("category"), 80) or "other",
                "summary": summary,
                "evidence": clean_text(item.get("evidence"), 700),
                "confidence": clean_text(item.get("confidence"), 40) or "medium",
                "chunk_index": chunk_index,
            }
        )
    show_md(f"Reviewed chunk **{chunk_index}/{len(line_chunks)}**; context items so far: **{len(context_items)}**")

# Simple dedupe by normalized category + summary.
seen_context_keys: set[tuple[str, str]] = set()
deduped_context_items: list[dict[str, Any]] = []
for item in context_items:
    key = (item["category"].lower(), re.sub(r"\W+", " ", item["summary"].lower()).strip())
    if key in seen_context_keys:
        continue
    seen_context_keys.add(key)
    deduped_context_items.append(item)

context_items = deduped_context_items[:MAX_CONTEXT_ITEMS]
show_md(f"## Context Extraction Complete\n- Context items retained: **{len(context_items)}**")
show_md(render_context_items_table(context_items))

## 6) Combine Context for Metadata Generation


In [ ]:
previous_ckan_context = compact_existing_dataset(previous_ckan_dataset)
if previous_ckan_context and is_blocked_dataset_url(previous_ckan_context.get("url")):
    previous_ckan_context["url"] = None

combined_context = {
    "notebook_inventory": notebook_inventory,
    "notebook_source_url": NOTEBOOK_SOURCE_URL,
    "blocked_dataset_urls": sorted(BLOCKED_DATASET_URLS),
    "context_items": context_items,
    "previous_ckan_entry": previous_ckan_context,
}

combined_context_text = json.dumps(combined_context, indent=2)
if len(combined_context_text) > MAX_COMBINED_CONTEXT_CHARS:
    combined_context_text = combined_context_text[:MAX_COMBINED_CONTEXT_CHARS - 3].rstrip() + "..."

show_md(f"""
## Combined Context
- Raw context items: **{len(context_items)}**
- Combined context characters sent to metadata LLM: **{len(combined_context_text)}**
""")

## 7) LLM Pass 2: Create CKAN Metadata


In [ ]:
METADATA_SYSTEM_PROMPT = """
You are generating CKAN package metadata for the output dataset or data product generated by a Jupyter notebook.
The notebook is evidence and provenance; the CKAN package should describe the produced dataset, map layer, file collection, API-backed data product, or analysis output.
Return STRICT JSON only. Do not include markdown, comments, or trailing commas.

Required JSON keys:
{
  "dataset_name": string,
  "dataset_title": string,
  "dataset_notes": string,
  "dataset_url": string or null,
  "dataset_author": string or null,
  "dataset_author_email": string or null,
  "dataset_maintainer": string or null,
  "dataset_maintainer_email": string or null,
  "dataset_license_id": string or null,
  "dataset_version": string or null,
  "dataset_type": string,
  "dataset_isopen": boolean,
  "dataset_spatial": GeoJSON Polygon object, GeoJSON MultiPolygon object, or null,
  "temporal_coverage_start": string or null,
  "temporal_coverage_end": string or null,
  "dataset_tags": list[string]
}

Rules:
- If previous_ckan_entry exists, preserve its dataset_name unless explicitly told otherwise.
- If preferred values are provided, use them unless contradicted by the notebook evidence.
- dataset_url should use the provided or inferred GitHub notebook source URL when available, unless the produced dataset has a more specific supported publication URL. Do not use unrelated metadata URLs from other registration workflows.
- Never use any URL listed in blocked_dataset_urls for dataset_url.
- dataset_name must be lowercase, URL-safe, hyphenated, and contain only letters, numbers, and hyphens.
- dataset_title should name the produced data product and include the location, measured quantity, units of measure, and method/source when the notebook evidence supports them. Example pattern: "Houston Area OPERA InSAR Vertical Displacement Rate (mm/year)". Do not invent any part of the title.
- Do not title the package as a notebook, tutorial, or workflow unless that is explicitly the data product.
- dataset_notes should be 2-4 concise sentences explaining what output dataset is generated, what source data it uses, the main transformation or method, and how the output may be reused. Mention the notebook only as provenance or generation method.
- Use evidence from the notebook source and output context. Do not invent authors, dates, license, spatial coverage, temporal coverage, location, units, or measured quantities.
- Temporal coverage must come from evidence in variables, constants, dataframe filters, filename date ranges, date-column min/max calculations, or printed/displayed output values. Prefer executed output prints/tables when they show the actual resulting date range.
- Use ISO-like dates for temporal_coverage_start and temporal_coverage_end when exact dates are available. If evidence only supports years, use the year values. If there is no supported temporal evidence, use null.
- Do not use notebook execution time, API request time, or metadata generation time as temporal coverage unless the output dataset itself is explicitly about that time.
- dataset_spatial must be a valid GeoJSON Polygon or MultiPolygon object in longitude/latitude order, not prose and not a bbox array.
- If a lon/lat bbox is the only spatial evidence, convert it to a closed GeoJSON Polygon ring using this order: [west,south], [west,north], [east,north], [east,south], [west,south]. Example: {"type":"Polygon","coordinates":[[[-96.27978518139572,28.08641391278623],[-96.27978518139572,30.71214633466123],[-93.65405275952072,30.71214633466123],[-93.65405275952072,28.08641391278623],[-96.27978518139572,28.08641391278623]]]}.
- If exact spatial geometry or bbox is not supported by notebook evidence, use null for dataset_spatial.
- If the notebook clearly references source data, output files, maps, APIs, models, organizations, location, or units, include them in notes and tags when they describe the produced dataset.
- Use null for unknown fields.
- Tags should be short lowercase discovery terms for the produced dataset, usually 6-15 items.
"""

metadata_payload = {
    "combined_context": json.loads(combined_context_text) if combined_context_text.endswith("}") else combined_context_text,
    "preferred_values": {
        "dataset_name": previous_ckan_dataset.get("name") if previous_ckan_dataset else existing_lookup_id or PREFERRED_DATASET_NAME,
        "dataset_title": PREFERRED_DATASET_TITLE,
        "dataset_url": NOTEBOOK_SOURCE_URL,
        "dataset_author": CKAN_DATASET_AUTHOR,
        "dataset_author_email": CKAN_DATASET_AUTHOR_EMAIL,
        "dataset_maintainer": CKAN_DATASET_MAINTAINER,
        "dataset_maintainer_email": CKAN_DATASET_MAINTAINER_EMAIL,
        "dataset_license_id": CKAN_DATASET_LICENSE_ID,
        "dataset_version": CKAN_DATASET_VERSION,
        "dataset_type": CKAN_DATASET_TYPE,
        "dataset_isopen": CKAN_DATASET_ISOPEN,
        "dataset_spatial": CKAN_DATASET_SPATIAL,
        "temporal_coverage_start": CKAN_TEMPORAL_COVERAGE_START,
        "temporal_coverage_end": CKAN_TEMPORAL_COVERAGE_END,
    },
}

llm_dataset = llm_json(METADATA_SYSTEM_PROMPT, metadata_payload, temperature=0.1)

def choose_text(key: str, fallback: str | None = None, max_chars: int | None = None) -> str:
    return clean_text(llm_dataset.get(key) or fallback, max_chars)


def choose_dataset_url(value: Any) -> str:
    candidate = clean_text(value)
    if is_blocked_dataset_url(candidate):
        candidate = ""
    if candidate:
        return candidate
    fallback = clean_text(NOTEBOOK_SOURCE_URL)
    if is_blocked_dataset_url(fallback):
        return ""
    return fallback


def bbox_to_geojson_polygon(bounds: list[Any]) -> dict[str, Any] | None:
    if len(bounds) != 4:
        return None
    try:
        west, south, east, north = [float(value) for value in bounds]
    except (TypeError, ValueError):
        return None
    return {
        "type": "Polygon",
        "coordinates": [[
            [west, south],
            [west, north],
            [east, north],
            [east, south],
            [west, south],
        ]],
    }


def normalize_geojson_spatial(value: Any, fallback: Any = None) -> str:
    candidate = value if value not in (None, "") else fallback
    if candidate in (None, ""):
        return ""

    parsed = candidate
    if isinstance(candidate, str):
        text = candidate.strip()
        if not text:
            return ""
        try:
            parsed = json.loads(text)
        except json.JSONDecodeError:
            numbers = [float(item) for item in re.findall(r"-?\d+(?:\.\d+)?", text)]
            if len(numbers) == 4 and re.search(r"\b(?:bbox|bounds?|extent)\b", text, re.IGNORECASE):
                parsed = bbox_to_geojson_polygon(numbers)
            else:
                return clean_text(text)

    if isinstance(parsed, dict):
        if "bbox" in parsed and isinstance(parsed["bbox"], list):
            polygon = bbox_to_geojson_polygon(parsed["bbox"])
            if polygon:
                parsed = polygon
        geojson_type = parsed.get("type")
        coordinates = parsed.get("coordinates")
        if geojson_type in {"Polygon", "MultiPolygon"} and coordinates:
            return json.dumps(parsed, separators=(",", ":"))
        return clean_text(json.dumps(parsed, separators=(",", ":")))

    if isinstance(parsed, list):
        polygon = bbox_to_geojson_polygon(parsed)
        if polygon:
            return json.dumps(polygon, separators=(",", ":"))
        return clean_text(json.dumps(parsed, separators=(",", ":")))

    return clean_text(parsed)


metadata_dataset_name = (
    previous_ckan_dataset.get("name") if previous_ckan_dataset else None
) or clean_text(existing_lookup_id) or clean_text(PREFERRED_DATASET_NAME) or slugify(choose_text("dataset_name") or choose_text("dataset_title") or notebook_inventory["title_hint"])
metadata_dataset_title = clean_text(PREFERRED_DATASET_TITLE) or choose_text("dataset_title", notebook_inventory["title_hint"], 160)
metadata_dataset_notes = choose_text("dataset_notes", f"Dataset generated by notebook {NOTEBOOK_PATH.name}.", 3000)

metadata_tags = dedupe_tags([tag for tag in llm_dataset.get("dataset_tags", []) if clean_text(tag)])

desired_dataset_payload = {
    "name": slugify(metadata_dataset_name),
    "title": metadata_dataset_title,
    "notes": metadata_dataset_notes,
    "url": choose_dataset_url(llm_dataset.get("dataset_url")),
    "owner_org": CKAN_OWNER_ORG_ID,
    "private": False,
    "author": choose_text("dataset_author", CKAN_DATASET_AUTHOR),
    "author_email": choose_text("dataset_author_email", CKAN_DATASET_AUTHOR_EMAIL),
    "maintainer": choose_text("dataset_maintainer", CKAN_DATASET_MAINTAINER),
    "maintainer_email": choose_text("dataset_maintainer_email", CKAN_DATASET_MAINTAINER_EMAIL),
    "license_id": choose_text("dataset_license_id", CKAN_DATASET_LICENSE_ID),
    "version": choose_text("dataset_version", CKAN_DATASET_VERSION),
    "type": choose_text("dataset_type", CKAN_DATASET_TYPE) or "dataset",
    "isopen": llm_dataset.get("dataset_isopen") if isinstance(llm_dataset.get("dataset_isopen"), bool) else CKAN_DATASET_ISOPEN,
    "spatial": normalize_geojson_spatial(llm_dataset.get("dataset_spatial"), CKAN_DATASET_SPATIAL),
    "temporal_coverage_start": choose_text("temporal_coverage_start", CKAN_TEMPORAL_COVERAGE_START),
    "temporal_coverage_end": choose_text("temporal_coverage_end", CKAN_TEMPORAL_COVERAGE_END),
    "tags": metadata_tags,
}

if is_blocked_dataset_url(desired_dataset_payload.get("url")):
    desired_dataset_payload["url"] = choose_dataset_url(None)

dataset_extras = [
    {"key": "source_notebook_path", "value": str(NOTEBOOK_PATH)},
    {"key": "source_notebook_name", "value": NOTEBOOK_PATH.name},
    {"key": "source_notebook_url", "value": NOTEBOOK_SOURCE_URL},
    {"key": "source_notebook_cell_count", "value": str(notebook_inventory["cell_count"])},
    {"key": "source_notebook_line_count", "value": str(notebook_inventory["source_line_count"])},
    {"key": "source_notebook_output_line_count", "value": str(notebook_inventory["output_line_count"])},
    {"key": "source_notebook_context_line_count", "value": str(notebook_inventory["line_count"])},
    {"key": "source_notebook_spatial_hint_count", "value": str(len(notebook_inventory["spatial_hints"]))},
    {"key": "source_notebook_unit_hint_count", "value": str(len(notebook_inventory["unit_hints"]))},
    {"key": "metadata_generated_from", "value": "notebook_ckan_registration.ipynb"},
    {"key": "metadata_generated_utc", "value": datetime.now(timezone.utc).isoformat()},
]

show_md("## Proposed CKAN Metadata")
show_md("```json\n" + json.dumps(desired_dataset_payload, indent=2) + "\n```")

## 8) Compare With Existing CKAN Metadata


In [ ]:
comparison_existing_dataset = previous_ckan_dataset or fetch_existing_dataset_or_none(CKAN_URL, desired_dataset_payload["name"], auth_header)
changes = compare_dataset_metadata(comparison_existing_dataset, desired_dataset_payload)

show_md(f"""
## Metadata Diff
- Existing dataset found: `{bool(comparison_existing_dataset)}`
- Dataset name: `{desired_dataset_payload['name']}`
- Fields needing create/update: **{sum(1 for row in changes if row['status'] != 'same')}**
""")
show_md(render_changes_table_markdown(changes))


## 9) Apply CKAN Create or Update

This cell writes to CKAN only when `APPLY_CHANGES=True`.


In [ ]:
dataset_after = comparison_existing_dataset
if APPLY_CHANGES:
    must_update = FORCE_DATASET_METADATA_UPDATE or comparison_existing_dataset is None or any(row["status"] != "same" for row in changes)
    dataset_extra_fields = {"url": desired_dataset_payload["url"]} if desired_dataset_payload.get("url") == "" else None
    if must_update:
        dataset_after = u.create_or_update_ckan_dataset(
            CKAN_URL,
            dataset_name=desired_dataset_payload["name"],
            dataset_title=desired_dataset_payload["title"],
            dataset_notes=desired_dataset_payload["notes"],
            dataset_tags=desired_dataset_payload["tags"],
            auth_header=auth_header,
            owner_org=CKAN_OWNER_ORG_ID,
            private=desired_dataset_payload["private"],
            dataset_author=desired_dataset_payload["author"],
            dataset_author_email=desired_dataset_payload["author_email"],
            dataset_maintainer=desired_dataset_payload["maintainer"],
            dataset_maintainer_email=desired_dataset_payload["maintainer_email"],
            dataset_license_id=desired_dataset_payload["license_id"],
            dataset_url=desired_dataset_payload["url"],
            dataset_version=desired_dataset_payload["version"],
            dataset_type=desired_dataset_payload["type"],
            dataset_isopen=desired_dataset_payload["isopen"],
            dataset_spatial=desired_dataset_payload["spatial"],
            temporal_coverage_start=desired_dataset_payload["temporal_coverage_start"],
            temporal_coverage_end=desired_dataset_payload["temporal_coverage_end"],
            dataset_extras=dataset_extras,
            extra_fields=dataset_extra_fields,
        )
        show_md(f"CKAN dataset create/update applied: `{dataset_after.get('name')}`")
    else:
        show_md("No CKAN dataset metadata changes needed.")
else:
    show_md("Dry run: CKAN dataset create/update skipped (`APPLY_CHANGES=False`).")

## 10) Optional Notebook Resource Upload

Set `UPLOAD_NOTEBOOK_RESOURCE=True` and `APPLY_CHANGES=True` to upload or update the notebook file itself as a CKAN resource.


In [ ]:
resource_created = 0
resource_updated = 0
if APPLY_CHANGES and UPLOAD_NOTEBOOK_RESOURCE:
    if dataset_after is None:
        raise ValueError("Cannot upload notebook resource because dataset_after is None.")
    resource_plan = [
        {
            "local_path": NOTEBOOK_PATH,
            "resource_name": NOTEBOOK_PATH.name,
            "resource_title": clean_text(desired_dataset_payload["title"], 180),
            "resource_description": clean_text(
                f"Source Jupyter notebook used to generate or describe this CKAN entry. "
                f"Notebook cells: {notebook_inventory['cell_count']}; source lines: {notebook_inventory['source_line_count']}; output lines: {notebook_inventory['output_line_count']}.",
                1000,
            ),
            "resource_tags": ["jupyter-notebook", "source-notebook"],
            "format": "IPYNB",
            "source_url": "",
        }
    ]
    uploaded, resource_created, resource_updated = upsert_resources(CKAN_URL, dataset_after, resource_plan, auth_header)
    show_md(f"Notebook resource uploaded/upserted: **{len(uploaded)}** (created={resource_created}, updated={resource_updated})")
elif APPLY_CHANGES:
    show_md("Notebook resource upload skipped (`UPLOAD_NOTEBOOK_RESOURCE=False`).")
else:
    show_md("Dry run: notebook resource upload skipped.")

## 11) Summary


In [ ]:
show_md(f"""
## Complete
- Notebook reviewed: `{NOTEBOOK_PATH}`
- Source lines walked: **{notebook_inventory['source_line_count']}**
- Output lines walked: **{notebook_inventory['output_line_count']}**
- Total context lines walked: **{notebook_inventory['line_count']}**
- Context items retained: **{len(context_items)}**
- CKAN dataset name: `{desired_dataset_payload['name']}`
- Existing dataset found before apply: `{bool(comparison_existing_dataset)}`
- APPLY_CHANGES: `{APPLY_CHANGES}`
- UPLOAD_NOTEBOOK_RESOURCE: `{UPLOAD_NOTEBOOK_RESOURCE}`
- Resource created: **{resource_created}**
- Resource updated: **{resource_updated}**
""")